# Visão Computacional com CNNs e Transformers
## 🎓 Faculdade Infnet — Pós-Graduação em Inteligência Artificial & Machine Learning
### Aula 3: Tradução Seq2Seq com `torch.nn.Transformer` Nativo do PyTorch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/allanspadini/curso-vision-transformers-infnet/blob/main/aula_03_fine_tuning_bert/aula_03_transformer_pytorch.ipynb)

---

### 🎯 Objetivo
Na Aula 2, implementamos todas as equações matemáticas do Transformer do zero.  
Aqui, utilizamos o módulo oficial **`torch.nn.Transformer`**, que instancia o **Encoder e o Decoder completos em uma única chamada**, resolvendo a mesma tarefa de tradução inglês-português com o mesmo dataset (`Helsinki-NLP/opus-100`).


### 1. Configuração do Ambiente e Verificação de GPU


In [ ]:
!pip install -q datasets tokenizers tqdm

import os
import math
import time
import random
import numpy as np
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers

# Reprodutibilidade e dispositivo
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔥 Executando em: {device}")


### 2. Dataset Oficial Opus-100 (`en-pt`) e Tokenização BPE

#### 🔴 A Situação-Problema:
Muitos tutoriais acadêmicos utilizam datasets minúsculos ou de domínio restrito (como *OpusBooks*, que contém apenas o livro *Alice no País das Maravilhas* com ~800 frases vitorianas). Ao tentar traduzir sentenças cotidianas no final do treinamento (ex: *'Talk to me'*, *'I have not lost yet'*), o modelo sofre com **erros severos de vocabulário fora da distribuição (OOV)** e alucinações repetitivas.

#### 🟢 A Solução de Engenharia:
Utilizamos o benchmark padrão da indústria **`Helsinki-NLP/opus-100`** (1.000.000 de pares em `en-pt`).
1. Selecionamos um subconjunto de **10.000 pares concisos (3 a 15 palavras)** para treinamento rápido e de alta densidade semântica.
2. Treinamos um tokenizador **Byte-Pair Encoding (BPE)** com 3.000 tokens diretamente nesse corpus moderno.
3. Separamos amostras do **split oficial de teste (`test`)** com a tradução esperada de referência (*Ground Truth*) para avaliar objetivamente a capacidade de generalização do modelo.


In [ ]:
# 1. Carregamento do Dataset Oficial Opus-100 (en-pt)
print("📥 Carregando dataset Opus-100 (en-pt) da Hugging Face...")
raw_dataset = load_dataset("Helsinki-NLP/opus-100", "en-pt")
print(f"✅ Pares disponíveis -> Treino: {len(raw_dataset['train']):,} | Teste: {len(raw_dataset['test']):,}")

# Filtrar frases concisas (3 a 15 palavras) para treino ágil e estável
train_filtered = raw_dataset["train"].select(range(30000)).filter(
    lambda ex: 3 <= len(ex["translation"]["en"].split()) <= 15 and 3 <= len(ex["translation"]["pt"].split()) <= 15
)
num_samples = 10000 if torch.cuda.is_available() else 1000
train_subset = train_filtered.select(range(min(num_samples, len(train_filtered))))

# Split oficial de teste com Ground Truth para avaliação cega
test_filtered = raw_dataset["test"].filter(
    lambda ex: 3 <= len(ex["translation"]["en"].split()) <= 10 and 3 <= len(ex["translation"]["pt"].split()) <= 10
)

# 2. Treinar Tokenizador BPE Compacto (3.000 tokens)
bpe_tokenizer = Tokenizer(models.BPE(unk_token="<UNK>"))
bpe_tokenizer.normalizer = normalizers.Lowercase()
bpe_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

SPECIAL_TOKENS = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
trainer = trainers.BpeTrainer(special_tokens=SPECIAL_TOKENS, vocab_size=3000, min_frequency=2)

all_texts = [ex["translation"]["en"] for ex in train_subset] + [ex["translation"]["pt"] for ex in train_subset]
bpe_tokenizer.train_from_iterator(all_texts, trainer=trainer)

PAD_IDX = bpe_tokenizer.token_to_id("<PAD>") # 0
SOS_IDX = bpe_tokenizer.token_to_id("<SOS>") # 1
EOS_IDX = bpe_tokenizer.token_to_id("<EOS>") # 2
VOCAB_SIZE = bpe_tokenizer.get_vocab_size()

# 3. Dataset e DataLoader
MAX_LEN = 30

class TranslationDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=30):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]["translation"]
        src = self.tokenizer.encode(item["en"]).ids[:self.max_len - 1] + [EOS_IDX]
        tgt = self.tokenizer.encode(item["pt"]).ids[:self.max_len - 1] + [EOS_IDX]
        
        src_pad = src + [PAD_IDX] * (self.max_len - len(src))
        tgt_pad = tgt + [PAD_IDX] * (self.max_len - len(tgt))
        return torch.tensor(src_pad, dtype=torch.long), torch.tensor(tgt_pad, dtype=torch.long)

train_loader = DataLoader(TranslationDataset(train_subset, bpe_tokenizer, MAX_LEN), batch_size=64, shuffle=True)
print(f"✅ Vocabulário BPE: {VOCAB_SIZE:,} tokens | Amostras de Treino: {len(train_subset):,} | Lotes: {len(train_loader):,}")


### 3. Modelo Seq2Seq com `torch.nn.Transformer` Nativo

> **💡 O PyTorch faz Positional Embeddings internamente?**  
> **Não.** Diferente de bibliotecas de alto nível como a Hugging Face (`transformers`), que recebem `input_ids` brutos e gerenciam embeddings de palavras e posições internamente, o **`torch.nn.Transformer` do PyTorch é puramente a espinha dorsal de autoatenção** — ele espera tensores de features contínuos `[Batch, Seq_Len, d_model]`.  
> Por ser invariante à ordem das sequências, **é mandatório** somar uma informação posicional aos embeddings dos tokens antes de passá-los para o `nn.Transformer`. Aqui usamos uma tabela aprendida `nn.Embedding(max_len, d_model)` (padrão em modelos como GPT e ViT).


In [ ]:
class Seq2SeqTransformer(nn.Module):
    """Modelo Seq2Seq enxuto construído diretamente sobre o torch.nn.Transformer."""
    def __init__(self, vocab_size: int, d_model: int = 256, nhead: int = 4,
                 num_layers: int = 3, dim_feedforward: int = 512, max_len: int = 100,
                 dropout: float = 0.1, pad_idx: int = 0):
        super().__init__()
        self.pad_idx = pad_idx
        self.d_model = d_model
        
        # 1. Embeddings de Token e de Posição (exigidos antes de entrar no nn.Transformer)
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.dropout = nn.Dropout(dropout)
        
        # 2. Instanciação do Encoder-Decoder COMPLETO em uma única linha!
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        
        # 3. Projeção final para os logits do vocabulário
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, src: torch.Tensor, tgt: torch.Tensor):
        device = src.device
        
        # Injeção de Posição
        pos_src = torch.arange(0, src.size(1), device=device).unsqueeze(0)
        pos_tgt = torch.arange(0, tgt.size(1), device=device).unsqueeze(0)
        
        src_vec = self.dropout(self.token_emb(src) * math.sqrt(self.d_model) + self.pos_emb(pos_src))
        tgt_vec = self.dropout(self.token_emb(tgt) * math.sqrt(self.d_model) + self.pos_emb(pos_tgt))
        
        # Máscaras de preenchimento (<PAD>) e causal (triangular superior)
        src_pad_mask = (src == self.pad_idx)
        tgt_pad_mask = (tgt == self.pad_idx)
        tgt_causal_mask = torch.triu(torch.ones(tgt.size(1), tgt.size(1), device=device, dtype=torch.bool), diagonal=1)
        
        # Processamento pelo Transformer nativo
        out = self.transformer(
            src=src_vec,
            tgt=tgt_vec,
            tgt_mask=tgt_causal_mask,
            src_key_padding_mask=src_pad_mask,
            tgt_key_padding_mask=tgt_pad_mask,
            memory_key_padding_mask=src_pad_mask
        )
        return self.fc_out(out)

model = Seq2SeqTransformer(vocab_size=VOCAB_SIZE, pad_idx=PAD_IDX).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"🚀 Modelo instanciado com {total_params:,} parâmetros treináveis!")


### 4. Treinamento Resiliente no Mundo Real (Teacher Forcing, Checkpointing & Pause)

#### 🔴 A Situação-Problema do Mundo Real:
Treinar Transformers é uma tarefa computacionalmente intensiva que pode levar horas ou dias. Em ambientes com instabilidade elétrica ou provedores de nuvem que desconectam sessões preemptivas (como o Google Colab), **uma queda aos 90% do treinamento (ex: na época 30) pode custar todo o progresso**, exigindo reiniciar do zero.

> **⚠️ Por que salvar apenas os pesos (`torch.save(model.state_dict(), ...)`) não é suficiente?**  
> 1. **Momentos do Otimizador (`AdamW`)**: O AdamW armazena internamente os momentos de primeira (`exp_avg`) e segunda ordem (`exp_avg_sq`) de cada parâmetro. Se você reiniciar sem restaurar o `optimizer.state_dict()`, o otimizador recomeça cego, podendo gerar gradientes instáveis e deteriorar o aprendizado já adquirido!  
> 2. **Histórico do Agendador (`CosineAnnealingLR`)**: O `scheduler` rastreia o contador de épocas para calcular a taxa de aprendizado atual ao longo da curva de decaimento do cosseno. Sem o `scheduler.state_dict()`, a taxa de aprendizado seria resetada para o valor máximo inicial (`8e-4`), destruindo o ajuste fino dos pesos na época 30!  
> 3. **Contador de Épocas & Melhores Métricas**: Precisamos saber exatamente qual foi a última época concluída (`epoch + 1`) e qual a melhor perda histórica (`best_loss`) para salvar o melhor modelo.

#### 🟢 A Solução de Engenharia:
1. **Dicionário de Checkpoint Completo**: Empacotamos e salvamos em disco o estado integral do treinamento (`model`, `optimizer`, `scheduler`, `epoch`, `loss`, `best_loss`).
2. **Retomada Automática (Auto-Resume)**: O código detecta se existe um checkpoint prévio em disco. Se houver, restaura os estados e continua da época seguinte sem qualquer intervenção manual!
3. **Pausa Graciosa (`KeyboardInterrupt`)**: Envolvemos o loop em um bloco `try ... except KeyboardInterrupt`. Se o usuário clicar em **Parar / Interromper Execução** no Colab/Jupyter ou pressionar `Ctrl+C` no terminal, o treinamento pausa com segurança e o último checkpoint permanece 100% íntegro.
4. **Persistência no Google Colab**: Se estiver no Colab, basta apontar o caminho para uma pasta no Google Drive (`/content/drive/MyDrive/...`) para que o checkpoint sobreviva mesmo se a máquina virtual for destruída!


In [ ]:
# --- 1. Configuração de Diretório e Arquivo de Checkpoint ---
import os

# DICA PRO: No Google Colab, descomente as duas linhas abaixo para salvar diretamente no Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# CHECKPOINT_DIR = '/content/drive/MyDrive/checkpoints_transformer'

CHECKPOINT_DIR = "./checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "transformer_seq2seq_last.pt")
BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "transformer_seq2seq_best.pt")

EPOCHS = 15 if torch.cuda.is_available() else 2
start_epoch = 1
best_loss = float('inf')

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.AdamW(model.parameters(), lr=8e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

# --- 2. Lógica de Retomada Automática (Auto-Resume) ---
if os.path.exists(CHECKPOINT_PATH):
    print(f"🔄 Checkpoint encontrado em: '{CHECKPOINT_PATH}'")
    print("   Restaurando estado completo (Modelo, Otimizador, Scheduler e Época)...")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    
    # Restauração integral dos estados
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    
    start_epoch = checkpoint['epoch'] + 1
    best_loss = checkpoint.get('best_loss', float('inf'))
    last_loss = checkpoint.get('loss', float('nan'))
    
    print(f"✅ Restauração concluída com sucesso!")
    print(f"   • Próxima época a executar: {start_epoch} de {EPOCHS}")
    print(f"   • Última perda registrada: {last_loss:.4f}")
    print(f"   • Melhor perda histórica:  {best_loss:.4f}")
    print(f"   • Taxa de aprendizado atual: {scheduler.get_last_lr()[0]:.6f}")
else:
    print("🚀 Nenhum checkpoint prévio encontrado. Iniciando treinamento do zero...")

# --- 3. Loop de Treinamento com Suporte a Pause (KeyboardInterrupt) ---
model.train()

if start_epoch > EPOCHS:
    print(f"✨ O treinamento já foi concluído até a época {EPOCHS}! Nenhum ciclo pendente.")
else:
    try:
        for epoch in range(start_epoch, EPOCHS + 1):
            epoch_loss = 0.0
            progress_bar = tqdm(train_loader, desc=f"Época {epoch:02d}/{EPOCHS:02d}", leave=False)

            for src, tgt in progress_bar:
                src, tgt = src.to(device), tgt.to(device)

                # Teacher Forcing: Decoder recebe [<SOS>, y_1, ..., y_{T-1}] e prevê [y_1, ..., y_T]
                sos_col = torch.full((tgt.size(0), 1), SOS_IDX, dtype=torch.long, device=device)
                decoder_input = torch.cat([sos_col, tgt[:, :-1]], dim=1)

                optimizer.zero_grad()
                logits = model(src, decoder_input)
                
                loss = criterion(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                epoch_loss += loss.item()
                progress_bar.set_postfix(loss=f"{loss.item():.4f}")

            scheduler.step()
            avg_loss = epoch_loss / len(train_loader)
            current_lr = scheduler.get_last_lr()[0]

            # Verificação de novo recorde de melhor perda
            is_best = avg_loss < best_loss
            if is_best:
                best_loss = avg_loss

            # Dicionário do Checkpoint com todas as variáveis críticas
            checkpoint_state = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'loss': avg_loss,
                'best_loss': best_loss
            }

            # Salvamento persistente do último estado e do melhor estado
            torch.save(checkpoint_state, CHECKPOINT_PATH)
            if is_best:
                torch.save(checkpoint_state, BEST_MODEL_PATH)

            best_flag = " ⭐ [Novo Melhor!]" if is_best else ""
            print(f"📍 Época [{epoch:02d}/{EPOCHS:02d}] | Perda: {avg_loss:.4f}{best_flag} | LR: {current_lr:.6f} | Checkpoint salvo 💾")

        print("\n🎉 Treinamento concluído com sucesso!")

    except KeyboardInterrupt:
        print("\n" + "="*70)
        print("⏸️ TREINAMENTO PAUSADO PELO USUÁRIO (Interrupção de Kernel / Ctrl+C)!")
        print(f"💾 O progresso até a última época finalizada está seguro em: '{CHECKPOINT_PATH}'")
        print("👉 Para continuar o treinamento exatamente de onde parou, basta reexecutar esta célula!")
        print("="*70)


### 4.1 Inspeção do Checkpoint Salvo em Disco
Podemos inspecionar o arquivo gerado para verificar os tensores e metadados salvos, comprovando a resiliência do estado:


In [ ]:
# Inspeção dos metadados e integridade do Checkpoint
if os.path.exists(CHECKPOINT_PATH):
    saved_ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
    print(f"📁 Arquivo de Checkpoint: {CHECKPOINT_PATH}")
    print(f"📦 Tamanho em Disco: {os.path.getsize(CHECKPOINT_PATH) / (1024 * 1024):.2f} MB")
    print("🔑 Chaves contidas no Checkpoint:", list(saved_ckpt.keys()))
    print(f"📍 Última Época Salva: {saved_ckpt['epoch']}")
    print(f"📉 Perda Registrada:   {saved_ckpt['loss']:.4f}")
    print(f"🏆 Melhor Perda:       {saved_ckpt['best_loss']:.4f}")
    print(f"⚙️ Quantidade de Tensores no Modelo: {len(saved_ckpt['model_state_dict'])}")
    print(f"🎯 Parâmetros rastreados no AdamW: {len(saved_ckpt['optimizer_state_dict']['state'])}")
else:
    print("ℹ️ Execute o treinamento primeiro para gerar o arquivo de checkpoint.")


### 5. Inferência Autorregressiva (Tradução na Prática)


In [ ]:
# Opcional: Se desejar carregar o melhor modelo histórico salvo pelo checkpoint:
if os.path.exists(BEST_MODEL_PATH):
    print(f"🏆 Carregando pesos do melhor modelo histórico ('{BEST_MODEL_PATH}') para inferência...")
    best_ckpt = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=False)
    model.load_state_dict(best_ckpt['model_state_dict'])

def translate(model, sentence: str, tokenizer, max_len: int = 30):
    """Tradução token a token usando a saída do modelo de forma autorregressiva."""
    model.eval()
    src_ids = tokenizer.encode(sentence).ids[:max_len - 1] + [EOS_IDX]
    src_tensor = torch.tensor(src_ids, dtype=torch.long).unsqueeze(0).to(device)

    tgt_indices = [SOS_IDX]

    for _ in range(max_len):
        tgt_tensor = torch.tensor(tgt_indices, dtype=torch.long).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = model(src_tensor, tgt_tensor)
        
        next_token = logits[0, -1].argmax().item()
        tgt_indices.append(next_token)
        
        if next_token == EOS_IDX:
            break

    output_tokens = [idx for idx in tgt_indices if idx not in [PAD_IDX, SOS_IDX, EOS_IDX]]
    return tokenizer.decode(output_tokens)

# Avaliação Objetiva: Amostras Vistas no Treino vs Amostras Inéditas do Teste com Ground Truth
print("🔍 --- 1. Amostras Vistas no Treino (Validação de Convergência) ---")
train_samples = [
    train_subset[10]["translation"],
    train_subset[25]["translation"]
]
for item in train_samples:
    pred = translate(model, item["en"], bpe_tokenizer)
    print(f"🇺🇸 EN:           {item['en']}")
    print(f"🎯 Esperado (PT): {item['pt']}")
    print(f"🤖 Previsto (PT): {pred}\n")

print("🔍 --- 2. Amostras Inéditas do Split de Teste (Generalização Real) ---")
test_samples = [
    test_filtered[7]["translation"],   # "I have not lost yet."
    test_filtered[15]["translation"],  # "I left him on the road."
    test_filtered[23]["translation"]   # "There are my boys."
]
for item in test_samples:
    pred = translate(model, item["en"], bpe_tokenizer)
    print(f"🇺🇸 EN:           {item['en']}")
    print(f"🎯 Esperado (PT): {item['pt']}")
    print(f"🤖 Previsto (PT): {pred}\n")
